# EMA Crossover Scanner — Swing-Low Stop Loss

Variant of `ema_strategy_sl_target_input.ipynb`. The **Stop Loss is determined by swing lows** instead of a fixed %, so SL varies per trade. Only the **Target %** remains a free parameter to optimize.

## Rules
- A **swing low** = a candle whose Low is the minimum within a `±SWING_WINDOW` pivot (default N=2 → 5-bar pivot).
- For each touch (entry) candle:
  - If the **touch candle itself is a swing low** → `SL = Low(touch candle)`.
  - Otherwise → `SL = Low(most recent prior swing low)`.
  - If no prior swing low exists → fall back to `Low(touch candle)`.
  - Trade is skipped if the resulting `SL ≥ Entry`.
- **Target %** is the only swept parameter; SL is *defined by structure*, not by %.
- Reports are produced **per (Timeframe, EMA Pair)** — 4 setups × 4 reports = 16 reports.

In [1]:
# ============================================================
#  INPUTS — Target % and Swing Window
# ============================================================
TARGET_PCT    = 5.0    # default target above Touch Close
SWING_WINDOW  = 2      # ± candles for pivot-low (N=2 → 5-bar pivot)

print('=' * 50)
print(f'  TARGET_PCT   = {TARGET_PCT}%')
print(f'  SWING_WINDOW = ±{SWING_WINDOW} candles')
print('=' * 50)

  TARGET_PCT   = 5.0%
  SWING_WINDOW = ±2 candles


In [2]:
# ============================================================
#  CONFIG
# ============================================================
STOCKS = ['RELIANCE', 'TCS', 'HDFCBANK', 'INFY', 'ICICIBANK']

FILE_MAP = {
    '1d': 'data/1d/{symbol}_historical.csv',
    '1h': 'data/1h/{symbol}_historical.csv',
}

SETUPS = [
    ['1d',  21, 50],
    ['1d',  10, 20],
    ['1h',  21, 50],
    ['1h',  10, 20],
]

DATA_FOLDER = '.'

In [3]:
import pandas as pd
import numpy as np
import os
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows',    300)
pd.set_option('display.width',       300)
pd.set_option('display.float_format', '{:.2f}'.format)

TIMEFRAMES = list(dict.fromkeys([s[0] for s in SETUPS]))
print(f'Stocks: {STOCKS}')
print(f'Setups: {SETUPS}')

Stocks: ['RELIANCE', 'TCS', 'HDFCBANK', 'INFY', 'ICICIBANK']
Setups: [['1d', 21, 50], ['1d', 10, 20], ['1h', 21, 50], ['1h', 10, 20]]


In [4]:
# ── LOAD CSV ───────────────────────────────────────────────────────────
def load_csv(symbol, interval):
    filename = FILE_MAP[interval].replace('{symbol}', symbol)
    filepath = os.path.join(DATA_FOLDER, filename)
    if not os.path.exists(filepath):
        raise FileNotFoundError(f'Not found: {filepath}')
    df = pd.read_csv(filepath)
    df.columns = [c.strip().lower() for c in df.columns]
    if 'datetime' in df.columns:
        df['Date'] = pd.to_datetime(df['datetime'])
    elif 'timestamp' in df.columns:
        df['Date'] = pd.to_datetime(df['timestamp'], unit='s') + pd.Timedelta(hours=5, minutes=30)
    else:
        raise ValueError(f'No datetime/timestamp column in {filename}')
    df = df.set_index('Date').sort_index()
    if df.index.tz is not None:
        df.index = df.index.tz_localize(None)
    df = df.rename(columns={'open':'Open','high':'High','low':'Low','close':'Close','volume':'Volume'})
    cols = [c for c in ['Open','High','Low','Close','Volume'] if c in df.columns]
    return df[cols].dropna()

print('load_csv() ready.')

load_csv() ready.


In [5]:
# ── EMA + SCAN ─────────────────────────────────────────────────────────
def add_emas(df, fast, slow):
    df = df.copy()
    df[f'EMA{fast}'] = df['Close'].ewm(span=fast, adjust=False).mean()
    df[f'EMA{slow}'] = df['Close'].ewm(span=slow, adjust=False).mean()
    return df

def bullish_cross_indices(df, fast, slow):
    above = df[f'EMA{fast}'] > df[f'EMA{slow}']
    cross = above & ~above.shift(1, fill_value=False)
    cross.iloc[0] = False
    return list(np.where(cross.values)[0])

def touch_zone(low, ef, es):
    return 'At/Below Slow EMA' if low <= es else 'Between Slow & Fast EMA'

def scan(stock, interval, fast, slow, df):
    df = add_emas(df, fast, slow)
    rows = []
    for cidx in bullish_cross_indices(df, fast, slow):
        cross_ts = df.index[cidx]
        for i in range(cidx + 1, len(df)):
            low, close, open_, high = (float(df['Low'].iloc[i]),
                                       float(df['Close'].iloc[i]),
                                       float(df['Open'].iloc[i]),
                                       float(df['High'].iloc[i]))
            ef = float(df[f'EMA{fast}'].iloc[i]); es = float(df[f'EMA{slow}'].iloc[i])
            if ef <= es:
                break
            if low <= ef and close > ef and close > open_:
                rows.append({
                    'Stock': stock, 'Timeframe': interval,
                    'EMA Pair': f'EMA{fast}/EMA{slow}',
                    'Cross Time': str(cross_ts),
                    'Touch Time': str(df.index[i]),
                    'Touch Open': round(open_, 2), 'Touch High': round(high, 2),
                    'Touch Low':  round(low,  2), 'Touch Close': round(close, 2),
                    f'EMA{fast} @ Touch': round(ef, 2),
                    f'EMA{slow} @ Touch': round(es, 2),
                    'Touch Zone': touch_zone(low, ef, es),
                    'Candles After Cross': i - cidx,
                })
    return rows

print('scan() ready.')

scan() ready.


In [6]:
# ── RUN SCANNER ────────────────────────────────────────────────────────
all_signals = []
stock_data  = {}

for setup in SETUPS:
    interval, fast, slow = setup
    for stock in STOCKS:
        try:
            df = load_csv(stock, interval)
            stock_data[(stock, interval)] = add_emas(df, fast, slow)
            sig = scan(stock, interval, fast, slow, df)
            all_signals.extend(sig)
            print(f'  {stock:10s} [{interval}] EMA{fast}/EMA{slow} → {len(sig)} signals')
        except Exception as e:
            print(f'  {stock} [{interval}] ERROR: {e}')

signals = pd.DataFrame(all_signals)
if not signals.empty:
    signals = signals.sort_values(['Stock','Timeframe','EMA Pair','Cross Time','Touch Time']).reset_index(drop=True)
    signals.index += 1

print(f'\nTotal signals: {len(signals)}')

  RELIANCE   [1d] EMA21/EMA50 → 24 signals
  TCS        [1d] EMA21/EMA50 → 19 signals
  HDFCBANK   [1d] EMA21/EMA50 → 40 signals
  INFY       [1d] EMA21/EMA50 → 22 signals
  ICICIBANK  [1d] EMA21/EMA50 → 36 signals
  RELIANCE   [1d] EMA10/EMA20 → 29 signals
  TCS        [1d] EMA10/EMA20 → 35 signals
  HDFCBANK   [1d] EMA10/EMA20 → 56 signals
  INFY       [1d] EMA10/EMA20 → 42 signals
  ICICIBANK  [1d] EMA10/EMA20 → 67 signals
  RELIANCE   [1h] EMA21/EMA50 → 17 signals
  TCS        [1h] EMA21/EMA50 → 9 signals
  HDFCBANK   [1h] EMA21/EMA50 → 5 signals
  INFY       [1h] EMA21/EMA50 → 17 signals
  ICICIBANK  [1h] EMA21/EMA50 → 13 signals
  RELIANCE   [1h] EMA10/EMA20 → 31 signals
  TCS        [1h] EMA10/EMA20 → 24 signals
  HDFCBANK   [1h] EMA10/EMA20 → 19 signals
  INFY       [1h] EMA10/EMA20 → 26 signals
  ICICIBANK  [1h] EMA10/EMA20 → 28 signals

Total signals: 559


## Swing Low Detection

A candle at index `i` is a swing low if `Low[i] == min(Low[i-N:i+N+1])` where `N = SWING_WINDOW`. This is a hindsight definition — for backtesting we can confirm pivots using future candles. For each `(stock, interval)` we pre-compute a boolean mask of swing-low candles.

In [7]:
# ── SWING LOW DETECTION ────────────────────────────────────────────────
def compute_swing_lows(df, window):
    """Return boolean array: True where df['Low'].iloc[i] is min of ±window window."""
    n = len(df)
    lows = df['Low'].values
    swing = np.zeros(n, dtype=bool)
    for i in range(window, n - window):
        if lows[i] == lows[i-window:i+window+1].min():
            # Strict pivot: also require it's lower than neighbours (to avoid flat zones)
            left  = lows[i-window:i].min()
            right = lows[i+1:i+window+1].min()
            if lows[i] < left and lows[i] < right:
                swing[i] = True
    return swing

def find_swing_sl(df, touch_idx, swing_mask):
    """Return (sl_price, source_label) for a given touch index."""
    if swing_mask[touch_idx]:
        return float(df['Low'].iloc[touch_idx]), 'touch is swing low'
    prior = np.where(swing_mask[:touch_idx])[0]
    if len(prior) > 0:
        s = prior[-1]
        return float(df['Low'].iloc[s]), f'prior swing low @ {df.index[s]}'
    return float(df['Low'].iloc[touch_idx]), 'no prior swing low → touch low'

# Pre-compute swing-low masks for every (stock, interval) DataFrame
swing_masks = {}
for (stock, interval), df in stock_data.items():
    swing_masks[(stock, interval)] = compute_swing_lows(df, SWING_WINDOW)

print('Swing-low masks built for', len(swing_masks), 'series.')
for key, mask in list(swing_masks.items())[:5]:
    print(f'  {key} → {mask.sum()} swing lows in {len(mask)} candles')

Swing-low masks built for 10 series.
  ('RELIANCE', '1d') → 76 swing lows in 582 candles
  ('TCS', '1d') → 78 swing lows in 582 candles
  ('HDFCBANK', '1d') → 85 swing lows in 582 candles
  ('INFY', '1d') → 73 swing lows in 582 candles
  ('ICICIBANK', '1d') → 80 swing lows in 582 candles


In [8]:
# ── EVALUATE WITH SWING-LOW SL + TARGET % ──────────────────────────────
def candles_to_duration(n, interval):
    if interval in ('60', '1h'): return f'{n}h'
    if interval == '1d':         return f'{n}d'
    return f'{n} candles'

def evaluate_swing(signals_df, stock_data, swing_masks, target_pct):
    out = []
    for _, row in signals_df.iterrows():
        stock    = row['Stock']
        interval = row['Timeframe']
        entry    = float(row['Touch Close'])
        touch_ts = pd.Timestamp(row['Touch Time'])

        df = stock_data.get((stock, interval))
        swing_mask = swing_masks.get((stock, interval))
        if df is None or swing_mask is None:
            continue
        try:
            idx = df.index.get_loc(touch_ts)
        except KeyError:
            idx = df.index.searchsorted(touch_ts)

        sl_price, sl_source = find_swing_sl(df, idx, swing_mask)
        # If SL is not below entry, the trade has no valid stop — skip
        if sl_price >= entry:
            continue
        sl_pct = (entry - sl_price) / entry * 100
        target_price = round(entry * (1 + target_pct/100), 2)

        target_hit = sl_hit = False
        target_idx = sl_idx = None
        target_time = sl_time = None
        max_high = float('-inf')
        for j in range(idx + 1, len(df)):
            hi = float(df['High'].iloc[j]); lo = float(df['Low'].iloc[j])
            max_high = max(max_high, hi)
            if hi >= target_price and not target_hit:
                target_hit = True; target_idx = j; target_time = df.index[j]
            if lo <= sl_price and not sl_hit:
                sl_hit = True; sl_idx = j; sl_time = df.index[j]
            if target_hit or sl_hit:
                break

        if target_hit and sl_hit:
            outcome = ('TARGET HIT' if target_idx < sl_idx
                       else 'SL HIT' if sl_idx < target_idx
                       else 'AMBIGUOUS (same candle)')
        elif target_hit:  outcome = 'TARGET HIT'
        elif sl_hit:      outcome = 'SL HIT'
        else:             outcome = 'OPEN'

        exit_idx = (target_idx if outcome == 'TARGET HIT'
                    else sl_idx if outcome == 'SL HIT'
                    else len(df) - 1)
        n_cdl = exit_idx - idx if exit_idx is not None else None
        duration = candles_to_duration(n_cdl, interval) if n_cdl is not None else None

        if outcome == 'TARGET HIT':  pnl_pct =  target_pct
        elif outcome == 'SL HIT':    pnl_pct = -sl_pct
        else:                        pnl_pct = (float(df['Close'].iloc[-1]) - entry) / entry * 100

        out.append({
            **row,
            'Entry':         round(entry, 2),
            'SL Price':      round(sl_price, 2),
            'SL %':          round(sl_pct, 2),
            'SL Source':     sl_source,
            'Target Price':  target_price,
            'R:R':           round(target_pct / sl_pct, 2) if sl_pct > 0 else None,
            'Outcome':       outcome,
            'SL Hit Time':     str(sl_time)     if sl_time     else 'Not Hit',
            'Target Hit Time': str(target_time) if target_time else 'Not Hit',
            'Max High Seen': round(max_high, 2) if max_high != float('-inf') else None,
            'PnL %':         round(pnl_pct, 2),
            'Duration':      duration,
        })
    return pd.DataFrame(out)

results = evaluate_swing(signals, stock_data, swing_masks, TARGET_PCT)
print(f'Evaluated {len(results)} signals at Target={TARGET_PCT}%')
print(f'(skipped {len(signals)-len(results)} signals where swing SL ≥ entry)')

Evaluated 558 signals at Target=5.0%
(skipped 1 signals where swing SL ≥ entry)


In [9]:
# ── OVERALL + PER-SETUP + PER-STOCK SUMMARY @ default Target ───────────
sep = '=' * 70
total = len(results)
tgt   = (results['Outcome'] == 'TARGET HIT').sum()
sl    = (results['Outcome'] == 'SL HIT').sum()
op    = (results['Outcome'] == 'OPEN').sum()
amb   = (results['Outcome'] == 'AMBIGUOUS (same candle)').sum()
decided = tgt + sl

print(sep)
print(f'  SWING-LOW SL @ Target={TARGET_PCT}%')
print(sep)
print(f'  Total signals      : {total}')
print(f'  TARGET HIT         : {tgt}  ({tgt/total*100:.1f}%)')
print(f'  SL HIT             : {sl}   ({sl/total*100:.1f}%)')
print(f'  OPEN               : {op}   ({op/total*100:.1f}%)')
if amb: print(f'  AMBIGUOUS          : {amb}')
if decided: print(f'  Win rate (decided) : {tgt/decided*100:.1f}%')
print(f'  Avg SL %           : {results["SL %"].mean():.2f}%')
print(f'  Avg R:R            : {results["R:R"].mean():.2f}')
print(f'  Avg PnL %          : {results["PnL %"].mean():.2f}%')
print(f'  Total PnL %        : {results["PnL %"].sum():.2f}%')
print(sep)

print('\nPer Setup:')
per_setup = (results.groupby(['Timeframe', 'EMA Pair'])
             .apply(lambda x: pd.Series({
                 'Total':       len(x),
                 'Target Hit':  (x['Outcome']=='TARGET HIT').sum(),
                 'SL Hit':      (x['Outcome']=='SL HIT').sum(),
                 'Open':        (x['Outcome']=='OPEN').sum(),
                 'Win Rate %':  round((x['Outcome']=='TARGET HIT').mean()*100, 1),
                 'Avg SL %':    round(x['SL %'].mean(), 2),
                 'Avg R:R':     round(x['R:R'].mean(), 2),
                 'Avg PnL %':   round(x['PnL %'].mean(), 2),
                 'Total PnL %': round(x['PnL %'].sum(), 2),
             })).reset_index())
display(per_setup)

print('\nPer Stock:')
per_stock = (results.groupby('Stock')
             .apply(lambda x: pd.Series({
                 'Total':       len(x),
                 'Target Hit':  (x['Outcome']=='TARGET HIT').sum(),
                 'SL Hit':      (x['Outcome']=='SL HIT').sum(),
                 'Win Rate %':  round((x['Outcome']=='TARGET HIT').mean()*100, 1),
                 'Avg SL %':    round(x['SL %'].mean(), 2),
                 'Avg PnL %':   round(x['PnL %'].mean(), 2),
             })).reset_index())
display(per_stock)

  SWING-LOW SL @ Target=5.0%
  Total signals      : 558
  TARGET HIT         : 130  (23.3%)
  SL HIT             : 424   (76.0%)
  OPEN               : 4   (0.7%)
  Win rate (decided) : 23.5%
  Avg SL %           : 2.03%
  Avg R:R            : 3.74
  Avg PnL %          : -0.18%
  Total PnL %        : -102.08%

Per Setup:


,Timeframe,EMA Pair,Total,Target Hit,SL Hit,Open,Win Rate %,Avg SL %,Avg R:R,Avg PnL %,Total PnL %
0,1d,EMA10/EMA20,229.00,59.00,169.00,1.00,25.80,2.43,2.69,-0.29,-65.80
1,1d,EMA21/EMA50,140.00,48.00,92.00,0.00,34.30,2.44,2.61,0.25,35.28
2,1h,EMA10/EMA20,128.00,16.00,110.00,2.00,12.50,1.23,6.01,-0.33,-42.27
3,1h,EMA21/EMA50,61.00,7.00,53.00,1.00,11.50,1.24,5.46,-0.48,-29.29



Per Stock:


,Stock,Total,Target Hit,SL Hit,Win Rate %,Avg SL %,Avg PnL %
0,HDFCBANK,120.00,33.00,87.00,27.50,1.89,0.15
1,ICICIBANK,143.00,26.00,116.00,18.20,1.94,-0.46
2,INFY,107.00,22.00,85.00,20.60,2.17,-0.56
3,RELIANCE,101.00,23.00,75.00,22.80,1.96,0.06
4,TCS,87.00,26.00,61.00,29.90,2.25,-0.01


In [10]:
# ── EXPORT TRADE TABLE @ DEFAULT TARGET ────────────────────────────────
signals_dir = 'signals_swing'
os.makedirs(signals_dir, exist_ok=True)
tag = f'tp{TARGET_PCT}_sw{SWING_WINDOW}'.replace('.', 'p')
out_path = os.path.join(signals_dir, f'trades_{tag}.csv')
results.to_csv(out_path, index=False)
print(f'Saved full trade table → {out_path}')

for (tf, pair), grp in results.groupby(['Timeframe', 'EMA Pair']):
    fname = f'trades_{tf}_{pair.replace("/","_")}_{tag}.csv'
    grp.to_csv(os.path.join(signals_dir, fname), index=False)
    print(f'  → {fname}  ({len(grp)} rows)')

Saved full trade table → signals_swing\trades_tp5p0_sw2.csv
  → trades_1d_EMA10_EMA20_tp5p0_sw2.csv  (229 rows)
  → trades_1d_EMA21_EMA50_tp5p0_sw2.csv  (140 rows)
  → trades_1h_EMA10_EMA20_tp5p0_sw2.csv  (128 rows)
  → trades_1h_EMA21_EMA50_tp5p0_sw2.csv  (61 rows)


## Per-EMA-Pair Target Optimization — 4 Reports Each

Sweeps **Target %** across a grid (SL is fixed per-trade by swing-low logic, so it's not a free parameter). For each of the 4 setups, produces:

| # | Report | What it tells you |
|---|---|---|
| 1 | Top 15 by **Avg PnL %**     | Best per-trade quality |
| 2 | Top 15 by **Total PnL %**   | Best absolute return |
| 3 | Top 15 by **Expectancy** (risk-adjusted view) | Best with Profit Factor / Drawdown / Calmar |
| 4 | **Per-Stock breakdown** at best Avg-PnL target | Which stocks drive the result |

In [11]:
# ── TARGET SWEEP GRID ──────────────────────────────────────────────────
TARGET_GRID = [round(x, 2) for x in np.arange(1.0, 20.01, 0.5)]   # 1% → 20% step 0.5
print(f'Target grid: {len(TARGET_GRID)} values from {TARGET_GRID[0]}% to {TARGET_GRID[-1]}%')

def evaluate_target_combo(sig_set, tp):
    """Return (trades_df, summary_dict) for a given target %."""
    r = evaluate_swing(sig_set, stock_data, swing_masks, tp).sort_values('Touch Time').reset_index(drop=True)
    pnl = r['PnL %'].astype(float)
    w = pnl[pnl > 0]; l = pnl[pnl < 0]
    tgt_n = (r['Outcome'] == 'TARGET HIT').sum()
    sl_n  = (r['Outcome'] == 'SL HIT').sum()
    op_n  = (r['Outcome'] == 'OPEN').sum()
    d = tgt_n + sl_n
    wr = len(w) / len(pnl) if len(pnl) else 0
    aw = w.mean() if len(w) else 0
    al = l.mean() if len(l) else 0
    exp = wr * aw + (1 - wr) * al
    pf = (w.sum() / abs(l.sum())) if l.sum() != 0 else np.inf
    eq = pnl.cumsum(); dd = (eq - eq.cummax()).min() if len(pnl) else 0
    calmar = (pnl.sum() / abs(dd)) if dd < 0 else np.inf
    summary = {
        'Target %':       tp,
        'Trades':         len(pnl),
        'Target Hit':     int(tgt_n),
        'SL Hit':         int(sl_n),
        'Open':           int(op_n),
        'Win Rate %':     round(tgt_n/d*100, 1) if d else 0.0,
        'Avg SL %':       round(r['SL %'].mean(), 2)  if len(r) else 0.0,
        'Avg R:R':        round(r['R:R'].mean(), 2)   if len(r) else 0.0,
        'Avg PnL %':      round(pnl.mean(), 3),
        'Total PnL %':    round(pnl.sum(), 2),
        'Avg Win %':      round(aw, 2),
        'Avg Loss %':     round(al, 2),
        'Expectancy %':   round(exp, 3),
        'Profit Factor':  round(pf, 2) if np.isfinite(pf) else np.nan,
        'Max Drawdown %': round(dd, 2),
        'Calmar':         round(calmar, 2) if np.isfinite(calmar) else np.nan,
    }
    return r, summary

print('evaluate_target_combo() ready.')

Target grid: 39 values from 1.0% to 20.0%
evaluate_target_combo() ready.


In [12]:
# ── PER-PAIR OPTIMIZATION + 4 REPORTS ──────────────────────────────────
setups_in_signals = signals[['Timeframe', 'EMA Pair']].drop_duplicates().values.tolist()
overall_winners = []
all_grids = {}

for tf, pair in setups_in_signals:
    sub = signals[(signals['Timeframe'] == tf) & (signals['EMA Pair'] == pair)].reset_index(drop=True)
    setup_tag   = f"{tf}_{pair.replace('/', '_')}"
    setup_label = f"{tf}  {pair}"

    print('\n' + '█' * 92)
    print(f'  SETUP: {setup_label}    ({len(sub)} signals)')
    print('█' * 92)

    if len(sub) == 0:
        print('  No signals. Skipping.')
        continue

    summaries = [evaluate_target_combo(sub, tp)[1] for tp in TARGET_GRID]
    grid = pd.DataFrame(summaries)
    grid.to_csv(os.path.join(signals_dir, f'opt_grid_{setup_tag}.csv'), index=False)
    all_grids[(tf, pair)] = grid

    # ── REPORT 1: top by Avg PnL ─────────────────────────────────────────
    print('\n  📊 REPORT 1 — Top 15 Targets by Avg PnL %')
    r1 = grid.sort_values('Avg PnL %', ascending=False).head(15).reset_index(drop=True)
    display(r1)
    best_avg = grid.sort_values('Avg PnL %', ascending=False).iloc[0]
    r1.to_csv(os.path.join(signals_dir, f'top_avg_{setup_tag}.csv'), index=False)

    # ── REPORT 2: top by Total PnL ───────────────────────────────────────
    print('\n  📈 REPORT 2 — Top 15 Targets by Total PnL %')
    r2 = grid.sort_values('Total PnL %', ascending=False).head(15).reset_index(drop=True)
    display(r2)
    best_tot = grid.sort_values('Total PnL %', ascending=False).iloc[0]
    r2.to_csv(os.path.join(signals_dir, f'top_total_{setup_tag}.csv'), index=False)

    # ── REPORT 3: risk-adjusted (Expectancy / PF / Calmar / DD visible) ──
    print('\n  ⚖️  REPORT 3 — Top 15 Targets by Expectancy (risk-adjusted view)')
    r3 = grid.sort_values('Expectancy %', ascending=False).head(15).reset_index(drop=True)
    display(r3)
    best_exp = grid.sort_values('Expectancy %', ascending=False).iloc[0]
    best_pf  = grid.dropna(subset=['Profit Factor']).sort_values('Profit Factor', ascending=False).iloc[0] if grid['Profit Factor'].notna().any() else best_exp
    best_cal = grid.dropna(subset=['Calmar']).sort_values('Calmar', ascending=False).iloc[0] if grid['Calmar'].notna().any() else best_exp
    r3.to_csv(os.path.join(signals_dir, f'top_risk_{setup_tag}.csv'), index=False)

    # ── REPORT 4: per-stock breakdown at best Avg-PnL target ─────────────
    print(f"\n  🏢 REPORT 4 — Per-stock breakdown at best Avg-PnL target (TP={best_avg['Target %']}%)")
    trades_best, _ = evaluate_target_combo(sub, float(best_avg['Target %']))
    trades_best.to_csv(os.path.join(signals_dir, f'trades_best_{setup_tag}.csv'), index=False)
    per_stock = (trades_best.groupby('Stock')
                 .apply(lambda x: pd.Series({
                     'Trades':     len(x),
                     'Target Hit': (x['Outcome']=='TARGET HIT').sum(),
                     'SL Hit':     (x['Outcome']=='SL HIT').sum(),
                     'Open':       (x['Outcome']=='OPEN').sum(),
                     'Win Rate %': round((x['Outcome']=='TARGET HIT').mean()*100, 1),
                     'Avg SL %':   round(x['SL %'].mean(), 2),
                     'Avg R:R':    round(x['R:R'].mean(), 2),
                     'Avg PnL %':  round(x['PnL %'].mean(), 2),
                     'Total PnL %':round(x['PnL %'].sum(), 2),
                 })).reset_index())
    display(per_stock)
    per_stock.to_csv(os.path.join(signals_dir, f'per_stock_best_{setup_tag}.csv'), index=False)

    # ── one-line winners summary ────────────────────────────────────────
    print(f"\n  🏆 WINNERS FOR {setup_label}:")
    print(f"     Best Avg PnL    : TP={best_avg['Target %']}%  Avg={best_avg['Avg PnL %']}%  "
          f"Tot={best_avg['Total PnL %']}%  WR={best_avg['Win Rate %']}%  AvgSL={best_avg['Avg SL %']}%")
    print(f"     Best Total PnL  : TP={best_tot['Target %']}%  Avg={best_tot['Avg PnL %']}%  "
          f"Tot={best_tot['Total PnL %']}%  WR={best_tot['Win Rate %']}%")
    print(f"     Best Expectancy : TP={best_exp['Target %']}%  Exp={best_exp['Expectancy %']}%  "
          f"DD={best_exp['Max Drawdown %']}%")
    print(f"     Best Profit Fct : TP={best_pf['Target %']}%   PF={best_pf['Profit Factor']}  "
          f"Tot={best_pf['Total PnL %']}%")
    print(f"     Best Calmar     : TP={best_cal['Target %']}%  Calmar={best_cal['Calmar']}  "
          f"DD={best_cal['Max Drawdown %']}%")

    overall_winners.append({
        'Timeframe': tf, 'EMA Pair': pair, 'Signals': len(sub),
        'Best Avg TP %':      best_avg['Target %'],
        'Best Avg PnL %':     best_avg['Avg PnL %'],
        'Best Total TP %':    best_tot['Target %'],
        'Best Total PnL %':   best_tot['Total PnL %'],
        'Best Exp TP %':      best_exp['Target %'],
        'Best Expectancy %':  best_exp['Expectancy %'],
        'Best PF TP %':       best_pf['Target %'],
        'Best Profit Factor': best_pf['Profit Factor'],
        'Best Calmar TP %':   best_cal['Target %'],
        'Best Calmar':        best_cal['Calmar'],
    })


████████████████████████████████████████████████████████████████████████████████████████████
  SETUP: 1d  EMA10/EMA20    (229 signals)
████████████████████████████████████████████████████████████████████████████████████████████

  📊 REPORT 1 — Top 15 Targets by Avg PnL %


,Target %,Trades,Target Hit,SL Hit,Open,Win Rate %,Avg SL %,Avg R:R,Avg PnL %,Total PnL %,Avg Win %,Avg Loss %,Expectancy %,Profit Factor,Max Drawdown %,Calmar
0,1.50,229,126,100,1,55.80,2.43,0.81,-0.07,-14.94,1.68,-2.26,-0.07,0.93,-50.43,-0.30
1,1.00,229,146,78,1,65.20,2.43,0.54,-0.11,-24.50,1.16,-2.40,-0.12,0.87,-59.48,-0.41
2,2.00,229,111,117,1,48.70,2.43,1.08,-0.12,-27.18,2.00,-2.13,-0.13,0.89,-54.36,-0.50
3,2.50,229,99,129,1,43.40,2.43,1.34,-0.12,-27.69,2.50,-2.13,-0.13,0.90,-52.14,-0.53
4,3.00,229,86,142,1,37.70,2.43,1.61,-0.18,-41.97,3.00,-2.11,-0.19,0.86,-57.64,-0.73
5,3.50,229,76,152,1,33.30,2.43,1.88,-0.25,-57.58,3.50,-2.13,-0.26,0.82,-77.51,-0.74
6,4.00,229,70,158,1,30.70,2.43,2.15,-0.27,-60.84,4.00,-2.16,-0.28,0.82,-82.66,-0.74
7,4.50,229,64,164,1,28.10,2.43,2.42,-0.28,-63.26,4.50,-2.14,-0.29,0.82,-80.66,-0.78
8,5.00,229,59,169,1,25.90,2.43,2.69,-0.29,-65.80,5.00,-2.13,-0.30,0.82,-81.43,-0.81
9,5.50,229,48,180,1,21.10,2.43,2.96,-0.52,-119.65,5.50,-2.13,-0.53,0.69,-137.78,-0.87



  📈 REPORT 2 — Top 15 Targets by Total PnL %


,Target %,Trades,Target Hit,SL Hit,Open,Win Rate %,Avg SL %,Avg R:R,Avg PnL %,Total PnL %,Avg Win %,Avg Loss %,Expectancy %,Profit Factor,Max Drawdown %,Calmar
0,1.50,229,126,100,1,55.80,2.43,0.81,-0.07,-14.94,1.68,-2.26,-0.07,0.93,-50.43,-0.30
1,1.00,229,146,78,1,65.20,2.43,0.54,-0.11,-24.50,1.16,-2.40,-0.12,0.87,-59.48,-0.41
2,2.00,229,111,117,1,48.70,2.43,1.08,-0.12,-27.18,2.00,-2.13,-0.13,0.89,-54.36,-0.50
3,2.50,229,99,129,1,43.40,2.43,1.34,-0.12,-27.69,2.50,-2.13,-0.13,0.90,-52.14,-0.53
4,3.00,229,86,142,1,37.70,2.43,1.61,-0.18,-41.97,3.00,-2.11,-0.19,0.86,-57.64,-0.73
5,3.50,229,76,152,1,33.30,2.43,1.88,-0.25,-57.58,3.50,-2.13,-0.26,0.82,-77.51,-0.74
6,4.00,229,70,158,1,30.70,2.43,2.15,-0.27,-60.84,4.00,-2.16,-0.28,0.82,-82.66,-0.74
7,4.50,229,64,164,1,28.10,2.43,2.42,-0.28,-63.26,4.50,-2.14,-0.29,0.82,-80.66,-0.78
8,5.00,229,59,169,1,25.90,2.43,2.69,-0.29,-65.80,5.00,-2.13,-0.30,0.82,-81.43,-0.81
9,6.00,229,45,183,1,19.70,2.43,3.23,-0.52,-119.55,6.00,-2.13,-0.53,0.69,-140.18,-0.85



  ⚖️  REPORT 3 — Top 15 Targets by Expectancy (risk-adjusted view)


...


  ⚖️  REPORT 3 — Top 15 Targets by Expectancy (risk-adjusted view)


,Target %,Trades,Target Hit,SL Hit,Open,Win Rate %,Avg SL %,Avg R:R,Avg PnL %,Total PnL %,Avg Win %,Avg Loss %,Expectancy %,Profit Factor,Max Drawdown %,Calmar
0,1.00,61,32,27,1,54.20,1.24,1.09,0.08,4.97,1.02,-1.07,0.06,1.17,-7.56,0.66
1,1.50,61,26,33,1,44.10,1.24,1.64,0.06,3.80,1.51,-1.12,0.04,1.10,-11.19,0.34
2,2.00,61,17,43,1,28.30,1.24,2.19,-0.27,-16.37,2.00,-1.17,-0.29,0.68,-28.78,-0.57
3,4.00,61,9,51,1,15.00,1.24,4.37,-0.40,-24.21,4.00,-1.18,-0.42,0.60,-40.94,-0.59
4,3.50,61,10,50,1,16.70,1.24,3.83,-0.40,-24.65,3.50,-1.19,-0.42,0.59,-40.94,-0.60
5,2.50,61,12,48,1,20.00,1.24,2.73,-0.42,-25.66,2.50,-1.16,-0.44,0.54,-40.94,-0.63
6,3.00,61,11,49,1,18.30,1.24,3.28,-0.42,-25.77,3.00,-1.20,-0.44,0.56,-40.94,-0.63
7,6.00,61,6,53,2,10.20,1.24,6.56,-0.42,-25.73,5.51,-1.21,-0.44,0.60,-40.94,-0.63
8,4.50,61,8,52,1,13.30,1.24,4.92,-0.43,-26.22,4.50,-1.20,-0.45,0.58,-40.94,-0.64
9,5.50,61,6,53,2,10.20,1.24,6.01,-0.47,-28.73,5.08,-1.21,-0.49,0.55,-40.94,-0.70



  🏢 REPORT 4 — Per-stock breakdown at best Avg-PnL target (TP=1.0%)


,Stock,Trades,Target Hit,SL Hit,Open,Win Rate %,Avg SL %,Avg R:R,Avg PnL %,Total PnL %
0,HDFCBANK,5.00,1.00,4.00,0.00,20.00,0.72,1.72,-0.31,-1.56
1,ICICIBANK,13.00,8.00,5.00,0.00,61.50,1.08,1.17,0.35,4.49
2,INFY,17.00,7.00,10.00,0.00,41.20,1.37,0.91,-0.19,-3.22
3,RELIANCE,17.00,9.00,6.00,1.00,52.90,1.18,1.24,0.04,0.61
4,TCS,9.00,7.00,2.00,0.00,77.80,1.67,0.70,0.52,4.65



  🏆 WINNERS FOR 1h  EMA21/EMA50:
     Best Avg PnL    : TP=1.0%  Avg=0.081%  Tot=4.97%  WR=54.2%  AvgSL=1.24%
     Best Total PnL  : TP=1.0%  Avg=0.081%  Tot=4.97%  WR=54.2%
     Best Expectancy : TP=1.0%  Exp=0.064%  DD=-7.56%
     Best Profit Fct : TP=1.0%   PF=1.17  Tot=4.97%
     Best Calmar     : TP=1.0%  Calmar=0.66  DD=-7.56%


In [13]:
# ── CONSOLIDATED WINNERS TABLE ─────────────────────────────────────────
print('═' * 92)
print('  CONSOLIDATED WINNERS — best Target per EMA setup, by each metric')
print('═' * 92)
winners_df = pd.DataFrame(overall_winners)
display(winners_df)
winners_df.to_csv(os.path.join(signals_dir, 'pair_winners_consolidated.csv'), index=False)
print(f'\nSaved → {os.path.join(signals_dir, "pair_winners_consolidated.csv")}')

# Global winners across all setups
print('\n' + '═' * 92)
print('  GLOBAL WINNERS (best across all 4 setups)')
print('═' * 92)
for metric_col in ['Best Avg PnL %', 'Best Total PnL %', 'Best Expectancy %', 'Best Profit Factor', 'Best Calmar']:
    if metric_col not in winners_df.columns:
        continue
    sub = winners_df.dropna(subset=[metric_col])
    if len(sub) == 0: continue
    b = sub.sort_values(metric_col, ascending=False).iloc[0]
    if metric_col == 'Best Avg PnL %':       tp_col = 'Best Avg TP %'
    elif metric_col == 'Best Total PnL %':   tp_col = 'Best Total TP %'
    elif metric_col == 'Best Expectancy %':  tp_col = 'Best Exp TP %'
    elif metric_col == 'Best Profit Factor': tp_col = 'Best PF TP %'
    else:                                    tp_col = 'Best Calmar TP %'
    print(f"  {metric_col:<22s}: {b['Timeframe']} {b['EMA Pair']}  TP={b[tp_col]}%  value={b[metric_col]}")

════════════════════════════════════════════════════════════════════════════════════════════
  CONSOLIDATED WINNERS — best Target per EMA setup, by each metric
════════════════════════════════════════════════════════════════════════════════════════════


,Timeframe,EMA Pair,Signals,Best Avg TP %,Best Avg PnL %,Best Total TP %,Best Total PnL %,Best Exp TP %,Best Expectancy %,Best PF TP %,Best Profit Factor,Best Calmar TP %,Best Calmar
0,1d,EMA10/EMA20,229,1.50,-0.07,1.50,-14.94,1.50,-0.07,1.50,0.93,1.50,-0.30
1,1d,EMA21/EMA50,141,4.50,0.37,4.50,51.74,4.50,0.37,1.00,1.67,1.00,2.75
2,1h,EMA10/EMA20,128,1.00,-0.16,1.00,-20.18,1.00,-0.16,1.00,0.73,4.00,-0.80
3,1h,EMA21/EMA50,61,1.00,0.08,1.00,4.97,1.00,0.06,1.00,1.17,1.00,0.66



Saved → signals_swing\pair_winners_consolidated.csv

════════════════════════════════════════════════════════════════════════════════════════════
  GLOBAL WINNERS (best across all 4 setups)
════════════════════════════════════════════════════════════════════════════════════════════
  Best Avg PnL %        : 1d EMA21/EMA50  TP=4.5%  value=0.37
  Best Total PnL %      : 1d EMA21/EMA50  TP=4.5%  value=51.74
  Best Expectancy %     : 1d EMA21/EMA50  TP=4.5%  value=0.37
  Best Profit Factor    : 1d EMA21/EMA50  TP=1.0%  value=1.67
  Best Calmar           : 1d EMA21/EMA50  TP=1.0%  value=2.75


### Files saved to `signals_swing/`

- `trades_tp{X}_sw{N}.csv` — full trade table at default Target.
- `trades_<tf>_<pair>_tp{X}_sw{N}.csv` — per-setup trade tables.
- `opt_grid_<tf>_<pair>.csv` — full Target sweep grid per setup.
- `top_avg_<tf>_<pair>.csv` — Report 1 (top 15 by Avg PnL).
- `top_total_<tf>_<pair>.csv` — Report 2 (top 15 by Total PnL).
- `top_risk_<tf>_<pair>.csv` — Report 3 (top 15 by Expectancy).
- `per_stock_best_<tf>_<pair>.csv` — Report 4 (per-stock breakdown).
- `trades_best_<tf>_<pair>.csv` — full trade rows at each setup's best target.
- `pair_winners_consolidated.csv` — one-row-per-setup winners across all 5 metrics.

**Tip:** change `TARGET_PCT` or `SWING_WINDOW` at top and rerun for sensitivity analysis.